# Plant Disease Classification
We train a model to classify diseases using resnet18.

In [ ]:
# 1. Imports and Setup
# Import libraries
!pip install -q datasets matplotlib pandas numpy torch torchvision scikit-learn
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import transforms, models
import datasets
from datasets import load_dataset, concatenate_datasets, DatasetDict
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from collections import Counter
import time
import copy

print(f"datasets version: {datasets.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# 2. Loading Dataset
# Load dataset and split it
print("Loading dataset")
dataset = load_dataset("BrandonFors/Plant-Diseases-PlantVillage-Dataset")
full_dataset = concatenate_datasets([dataset["train"], dataset["test"]])

print("Creating splits")
train_testvalid = full_dataset.train_test_split(test_size=0.3, seed=42)
test_valid = train_testvalid['test'].train_test_split(test_size=0.5, seed=42)

custom_dataset = DatasetDict({
    'train': train_testvalid['train'],
    'validation': test_valid['train'],
    'test': test_valid['test']
})

train_ds = custom_dataset["train"]
val_ds = custom_dataset["validation"]
test_ds = custom_dataset["test"]

print(f"Training set: {len(train_ds)}")
print(f"Validation set: {len(val_ds)}")
print(f"Test set: {len(test_ds)}")

feature_label = train_ds.features['label']
if hasattr(feature_label, 'names'):
    class_names = feature_label.names
else:
    class_names = sorted(list(set(train_ds['label'])))
print(f"Number of classes: {len(class_names)}")
num_classes = len(class_names)

In [ ]:
# 3. Class Weights
# Compute weights to handle class imbalance
labels = [sample['label'] for sample in train_ds]
counts = Counter(labels)

total_samples = len(labels)
class_weights = [total_samples / (num_classes * counts[i]) for i in range(num_classes)]
class_weights_tensor = torch.FloatTensor(class_weights).to(device)
print("Class weights assigned")

In [ ]:
# 4. Data Augmentation
# Add rotation and flips to prevent overfitting
transform_train = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

transform_test = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

def apply_transforms_train(examples):
    pixel_values = [transform_train(image.convert("RGB")) for image in examples['image']]
    return {"pixel_values": pixel_values, "label": examples["label"]}

def apply_transforms_test(examples):
    pixel_values = [transform_test(image.convert("RGB")) for image in examples['image']]
    return {"pixel_values": pixel_values, "label": examples["label"]}

train_ds.set_transform(apply_transforms_train)
val_ds.set_transform(apply_transforms_test)
test_ds.set_transform(apply_transforms_test)

batch_size = 64
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)
print("DataLoaders ready")

In [ ]:
# 5. Training Loop
def train_model(model, num_epochs=5):
    print("Training started")
    
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)
    
    model = model.to(device)
    
    criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch+1}/{num_epochs}')
        model.train()
        running_loss = 0.0
        
        for i, batch in enumerate(train_loader):
            inputs = batch['pixel_values'].to(device)
            labels = batch['label'].to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)

        epoch_loss = running_loss / len(train_ds)

        model.eval()
        running_corrects = 0

        with torch.no_grad():
            for batch in val_loader:
                inputs = batch['pixel_values'].to(device)
                labels = batch['label'].to(device)

                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
                running_corrects += torch.sum(preds == labels.data)

        epoch_acc = running_corrects.float() / len(val_ds) * 100
        print(f'Loss: {epoch_loss:.4f} Acc: {epoch_acc:.2f}%')

        if epoch_acc > best_acc:
            best_acc = epoch_acc
            best_model_wts = copy.deepcopy(model.state_dict())

    model.load_state_dict(best_model_wts)
    return model, best_acc.item()

In [ ]:
# 6. Model Training
# Train resnet
num_epochs = 5
resnet = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
resnet, acc_resnet = train_model(resnet, num_epochs)

torch.save(resnet.state_dict(), 'best_plant_disease_model.pth')
print("Model saved")

In [ ]:
# 7. Final Test
# Check against test set
resnet.eval()
test_corrects = 0

with torch.no_grad():
    for batch in test_loader:
        inputs = batch['pixel_values'].to(device)
        labels = batch['label'].to(device)

        outputs = resnet(inputs)
        _, preds = torch.max(outputs, 1)
        test_corrects += torch.sum(preds == labels.data)

test_acc = test_corrects.float() / len(test_ds) * 100
print(f"Test accuracy: {test_acc:.2f}%")